<a href="https://colab.research.google.com/github/afngh/transformers/blob/main/transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from rich.progress import track
import urllib.request
import math

In [2]:
class Embedding(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000, dropout: float=0.3):
    super().__init__()

    self.embed = nn.Embedding(vocab_size, dims)
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    return self.dropout(self.embed(x))

In [3]:
class PositionalEmbedding(nn.Module):
  def __init__(self, dims: int=64, vocab_size: int=5000):
    super().__init__()

    pe = torch.zeros(vocab_size, dims)

    position = torch.arange(0, vocab_size, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, dims, 2).float() * (-math.log(10000.0) / dims))

    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    self.pe = pe.unsqueeze(0)

  def forward(self, x):
    x = x + self.pe[:, :x.size(1)]
    return x

In [4]:
class Attention(nn.Module):
  def __init__(self, dims: int=64, dropout: float=0.3, head: int=4):
    super().__init__()

    self.dims = dims
    self.head = head
    self.hdim = dims // head

    self.dropout = nn.Dropout(dropout)

    self.q = nn.Linear(dims, dims)
    self.k = nn.Linear(dims, dims)
    self.v = nn.Linear(dims, dims)

    self.projection = nn.Linear(dims, dims)

  def forward(self,x):

    B, S, D = x.size()

    q = self.q(x)
    k = self.k(x)
    v = self.v(x)

    q = q.view(B, S, self.head, self.hdim).transpose(1, 2)
    k = k.view(B, S, self.head, self.hdim).transpose(1, 2)
    v = v.view(B, S, self.head, self.hdim).transpose(1, 2)

    scores = torch.matmul(q,k.transpose(-1,-2) / self.hdim ** 0.5)

    attention_weights = torch.softmax(scores, dim=-1)
    attention_weights = self.dropout(attention_weights)

    x = torch.matmul(attention_weights, v)
    x = x.transpose(1, 2).contiguous().view(B, S, D)
    x = self.projection(x)

    return x

In [13]:
class PostAttention(nn.Module):
  def __init__(self, dims: int=64, dropout: float=0.3):
    super().__init__()

    self.norm1 = nn.LayerNorm(dims)
    self.norm2 = nn.LayerNorm(dims)

    self.fnn = nn.Sequential(
        nn.Linear(dims, dims*4),
        nn.ReLU(),
        nn.Linear(dims*4, dims)
    )

    self.drop1 = nn.Dropout(dropout)
    self.drop2 = nn.Dropout(dropout)

  def forward(self, x, attention):
    x = x + self.drop1(attention)
    x = self.norm1(x)

    f = self.fnn(x)

    x = x + self.drop2(f)
    x = self.norm2(x)

    return x

In [9]:
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "text.txt")
with open("text.txt") as f:
    text = f.read(5000)

words = text.split()

vocab = sorted(list(set(words)))

wti = {word: i for i, word in enumerate(vocab)}
itw = {i: word for i, word in enumerate(vocab)}

seq = 5

X = []
y = []

for i in range(len(words)-seq):
  X.append([wti[w] for w in words[i:i+seq]])
  y.append([wti[words[i+seq]]])

X = torch.tensor(X)
y = torch.tensor(y)

x_test = torch.tensor([[21,  15,  10, 450]])
y_test = torch.tensor([[332]])

In [15]:
embeddings = Embedding(vocab_size=len(vocab))
pos_embeddings = PositionalEmbedding(vocab_size=len(vocab))
attention = Attention()

word_embedding = embeddings(x_test)
input = pos_embeddings(word_embedding)
# print(input.shape)
encoder_output = PostAttention(dropout=0.3)

attention_output = attention(input)
encoder_output(input, attention_output)

tensor([[[ 1.6956e-01,  9.2280e-01, -2.0494e+00, -7.2612e-02, -3.8035e-01,
          -5.4552e-01, -9.0377e-02,  6.5719e-01, -5.4240e-01,  9.5930e-02,
          -1.5504e+00, -2.0197e-01, -1.0557e+00,  1.4276e+00, -1.0093e+00,
           2.4829e+00,  4.6182e-02,  1.0912e+00,  1.0959e+00, -1.4329e-01,
          -6.9275e-01,  1.3517e+00, -9.3354e-01,  1.0143e+00, -1.2119e+00,
           1.3971e+00,  1.5477e+00,  3.4717e-01, -1.7194e+00, -7.1926e-02,
           5.5123e-01,  4.7631e-01, -1.2782e-02, -2.2382e+00,  7.1133e-01,
          -1.2825e+00, -1.6811e+00, -9.2059e-02, -4.9031e-01,  7.5118e-01,
           6.5389e-02,  3.1300e-01,  3.5243e-01, -3.2738e-01, -1.9751e-01,
           1.6491e-01, -1.2033e+00,  9.4938e-01, -6.4126e-01,  5.5344e-01,
           6.8996e-01, -2.0953e-01, -2.7454e+00,  3.9475e-01, -1.8122e-01,
           1.1348e+00, -4.3622e-01,  1.3584e+00,  4.4231e-01,  8.5369e-01,
           3.2494e-01,  1.2277e+00, -6.6491e-01, -2.8784e-01],
         [ 2.8463e-01,  2.9050e-01, -